# 🕳️ Reddit RabbitHole RAG v2 — Kaggle runbook

**Before you start (right sidebar → Session options):**
- **Internet: On** (needed for HuggingFace, Arctic Shift, GitHub)
- **Accelerator: GPU T4 x2 or P100** (only the *index* step needs it; ingest/meta/extract run fine on it too)
- **Add-ons → Secrets:** add `HF_TOKEN` (write access) and `GEMINI_API_KEY`. Attach both to this notebook.

**How to run:** go top to bottom. Section A is a 5-minute smoke test on a small slice — do it first. Section B is the full run (long). For the long steps, use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.

## 0. Setup

In [1]:
# ---- EDIT THESE ----------------------------------------------------------------
GITHUB_REPO   = "https://github.com/athxrvxaa/reddit-rabbithole-rag.git"
BRANCH        = "v2"
SOURCE        = "github"   # "github" = clone the branch above | "zip" = use the zip uploaded as a Kaggle Dataset
ARTIFACTS_REPO = "athxrvxaa/rabbithole-artifacts"     # PUBLIC HF dataset repo the Space downloads from
CKPT_REPO      = "athxrvxaa/rabbithole-checkpoints"   # PRIVATE HF dataset repo for the big ingest DB checkpoint
SMOKE_MAX_COMMENTS = 100_000
# --------------------------------------------------------------------------------

import os, glob, subprocess
os.chdir("/kaggle/working")

if SOURCE == "github":
    if not os.path.exists("reddit-rabbithole-rag"):
        subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_REPO, "reddit-rabbithole-rag"], check=True)
else:
    zips = glob.glob("/kaggle/input/**/reddit-rabbithole-rag-v2.zip", recursive=True)
    assert zips, "Add the zip as a Kaggle Dataset (Add Input) first."
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", "/kaggle/working"], check=True)

os.chdir("/kaggle/working/reddit-rabbithole-rag")
print("repo at", os.getcwd())

Cloning into 'reddit-rabbithole-rag'...


repo at /kaggle/working/reddit-rabbithole-rag


In [2]:
!pip install -q -r requirements-pipeline.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 31.0 MB/s eta 0:00:00


In [3]:
# Load secrets into the environment (subprocesses started with ! inherit them)
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for name in ("HF_TOKEN", "GEMINI_API_KEY"):
    try:
        os.environ[name] = sec.get_secret(name)
        print(f"{name}: loaded")
    except Exception:
        print(f"{name}: MISSING (add it under Add-ons -> Secrets and attach it to this notebook)")

os.environ["ARTIFACTS_REPO"] = ARTIFACTS_REPO
os.environ["RR_ARTIFACTS_DIR"] = "/kaggle/working/artifacts"

import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

HF_TOKEN: loaded
GEMINI_API_KEY: loaded
GPU available: True Tesla T4


In [4]:
# Sanity: the code's own tests (takes a few seconds)
!python -m pytest -q

..................................                                       [100%]
34 passed in 13.01s


## A. Smoke test (≈5–10 min) — do this first
Streams only the first rows into a **separate** folder, so it never mixes with the full run.
Check the output of `stats`: `op_source` should be almost all `submission`, answers should look sensible, categories not all `other`.

In [5]:
os.environ["RR_DATA_DIR"] = "/kaggle/working/data_smoke"
!python -m rabbithole ingest --max-comments {SMOKE_MAX_COMMENTS}

Streaming HuggingFaceGECLM/REDDIT_comments [IAmA] (resuming at row 0; thread sample rate 1.0; 0 rows already stored)
README.md: 100%|███████████████████████████| 7.57k/7.57k [00:00<00:00, 2.78MB/s]
Resolving data files: 100%|██████████████████| 46/46 [00:00<00:00, 23372.26it/s]
  scanned 20,000 rows (1,200/s) | kept 14,283
  scanned 40,000 rows (1,623/s) | kept 28,381
  scanned 60,000 rows (1,858/s) | kept 42,458
  scanned 80,000 rows (2,002/s) | kept 56,601
  scanned 100,000 rows (2,106/s) | kept 71,038
Indexing comments by thread...
Done: 71,038 comments across 1,842 threads stored in /kaggle/working/data_smoke/rabbithole.db


In [6]:
!python -m rabbithole meta --min-comments 5

1,513 threads (>= 5 comments) need metadata
  50/1,513 done (found 50, missing 0, failed 0)
  1,050/1,513 done (found 1,050, missing 0, failed 0)
Finished. found=1,513 missing=0 failed(will retry on re-run)=0


In [7]:
!python -m rabbithole extract
!python -m rabbithole stats

1,513 threads have submission metadata
  500 threads -> pairs so far
{
  "threads_seen": 1842,
  "threads_no_pairs": 538,
  "threads_with_pairs": 734,
  "threads_too_small": 570
}
Extracted 6,334 unique Q&A pairs
pairs: 6,334
threads: 734
op_source: {'submission': 6334}
categories:
  other                     2,310  (36.5%)
  identity_lifestyle          529  (8.4%)
  health_medicine             525  (8.3%)
  relationships_family        508  (8.0%)
  education                   420  (6.6%)
  religion                    338  (5.3%)
  business_finance            287  (4.5%)
  crime_law                   251  (4.0%)
  jobs_trades                 195  (3.1%)
  entertainment_media         193  (3.0%)
  gaming_sports               168  (2.7%)
  politics_government         164  (2.6%)
  travel_adventure            138  (2.2%)
  tech                        138  (2.2%)
  military                    138  (2.2%)
  science_space                32  (0.5%)
answer chars p10/p50/p90/max: 121 358 1148 3

In [8]:
# Eyeball a few pairs: do questions look like real questions and answers like real answers?
import sqlite3, textwrap
con = sqlite3.connect("/kaggle/working/data_smoke/rabbithole.db"); con.row_factory = sqlite3.Row
for r in con.execute("SELECT title, question, answer, op_source FROM pairs ORDER BY RANDOM() LIMIT 5"):
    print("AMA :", r["title"][:90], f"[{r['op_source']}]")
    print("Q   :", textwrap.shorten(r["question"], 200))
    print("A   :", textwrap.shorten(r["answer"], 250), "\n")

AMA : I gave a child up for adoption when I was 17. AMA [submission]
Q   : Adopted child here. I realize the situation is different for every parent and every child, but do you think there is always a time to seek birth parents out? Assuming the adoption was under [...]
A   : What do you mean by always a time? I don't think that seeking out your birth parents is a bad thing, and I think it's probably healthy for an adopted child. Do you mean is there a specific time that's best to seek them out? 

AMA : I have heterochromia. AMA. [submission]
Q   : ok I'll start...do you feel like you treated differently / people stare at you?
A   : Occasionally people I don't know very well will look at me and say, "did you know your eye is two different colors?!" It's like when you get a haircut, and people the next day say, "hey, you got a haircut!" Reminds me of Douglas Adams: Other [...] 

AMA : I have Crohn's Disease. Ask me (almost) anything. [submission]
Q   : Did you find the diagnosis and ear

**If this looks wrong, stop here** and fix `rabbithole/extract.py` (`ExtractConfig`) before the full run. Common knobs: `min_answer_score`, `max_pairs_per_thread`, `min_thread_comments`.
Common problems: `datasets` script errors → `!pip install "datasets<4"`; Arctic Shift 429s → just re-run the `meta` cell.

## B. Full run
Everything below uses `/kaggle/working/data` (separate from the smoke test).

In [9]:
os.environ["RR_DATA_DIR"] = "/kaggle/working/data"

# Optional: ONLY if the Kaggle session restarted mid-ingest and you uploaded a checkpoint earlier
RESTORE_CHECKPOINT = False
if RESTORE_CHECKPOINT:
    from huggingface_hub import hf_hub_download
    os.makedirs("/kaggle/working/data", exist_ok=True)
    hf_hub_download(CKPT_REPO, "rabbithole.db", repo_type="dataset", local_dir="/kaggle/working/data", token=os.environ["HF_TOKEN"])
    print("restored checkpoint")

### B0. Pick a sample (the dataset is huge — you only need ~4–5M comments)
The cell below peeks at the source and works out a **thread sample rate**: it keeps a random but fixed ~X% of *whole threads* (so Q&A pairs aren't cut in half), instead of just the first N rows.

- It reads the split size from HF metadata. If that fails, type the IAmA split's row count into `SPLIT_ROWS` yourself (HF dataset page → split dropdown → IAmA).
- Sampling needs a full scan of the split. At ~50k rows/s that's `rows / 50k / 60` minutes — if the split is in the hundreds of millions, set `FAST_FIRST_N = True` to just take the first `TARGET_ROWS` rows instead (faster, but covers only the start of the file).

In [10]:
TARGET_ROWS   = 5_000_000   # comments to keep (4-5M is plenty)
SPLIT_ROWS    = None        # None = auto-detect; or type the IAmA split's row count
FAST_FIRST_N  = False       # True = skip the full scan, take the first TARGET_ROWS rows

!python -m rabbithole inspect-source --target-rows {TARGET_ROWS}

from rabbithole.config import get_settings
from rabbithole.source_info import split_size, suggest_rate
if SPLIT_ROWS is None:
    SPLIT_ROWS = split_size(get_settings())

if FAST_FIRST_N or not SPLIT_ROWS:
    INGEST_ARGS = f"--max-comments {TARGET_ROWS}"
    print("\nStrategy: first-N slice ->", INGEST_ARGS,
          "" if FAST_FIRST_N else "(split size unknown; set SPLIT_ROWS to use thread sampling instead)")
else:
    RATE = suggest_rate(SPLIT_ROWS, TARGET_ROWS)
    INGEST_ARGS = f"--thread-sample-rate {RATE} --target-rows {int(TARGET_ROWS * 1.3)}"
    print(f"\nStrategy: thread sampling ->", INGEST_ARGS)
    print(f"Expect ~{int(SPLIT_ROWS * RATE):,} kept rows; full scan ~{SPLIT_ROWS / 50_000 / 60:,.0f} min at 50k rows/s")

Reading the first 20,000 rows of HuggingFaceGECLM/REDDIT_comments [IAmA] ...
Resolving data files: 100%|██████████████████| 46/46 [00:00<00:00, 25467.00it/s]
columns: ['archived', 'author', 'author_fullname', 'body', 'comment_type', 'controversiality', 'created_utc', 'edited', 'gilded', 'id', 'link_id', 'locked', 'name', 'parent_id', 'permalink', 'retrieved_on', 'score', 'subreddit_id', 'subreddit_name_prefixed', 'subreddit_type', 'total_awards_received']
  rows_checked: 20000
  distinct_threads: 552
  same_thread_as_previous_row: 0.263
  time_nondecreasing: 1.0
  time_range: ('2009-05-28', '2009-07-11')
  avg_body_chars: 258
  layout: time_ordered
  advice: Rows are ordered by time with threads interleaved. --max-comments N would give one time slice full of half-cut threads. Use --thread-sample-rate (whole threads, whole timeline).
Resolving data files: 100%|██████████████████| 46/46 [00:00<00:00, 21160.12it/s]

split rows: 25,778,822  ->  for ~5,000,000 kept rows use --thread-sample-

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]


Strategy: thread sampling -> --thread-sample-rate 0.19396 --target-rows 6500000
Expect ~5,000,060 kept rows; full scan ~9 min at 50k rows/s


### B1. Ingest the sample — the long one (resumable; re-run the cell if it stops)
Keep the same settings when resuming (the DB remembers the sample rate and refuses a mismatch).

In [11]:
!python -m rabbithole ingest {INGEST_ARGS}

Streaming HuggingFaceGECLM/REDDIT_comments [IAmA] (resuming at row 0; thread sample rate 0.19396; 0 rows already stored)
Resolving data files: 100%|██████████████████| 46/46 [00:00<00:00, 22819.39it/s]
  scanned 20,000 rows (1,329/s) | kept 3,110
  scanned 40,000 rows (1,783/s) | kept 5,485
  scanned 60,000 rows (2,000/s) | kept 8,669
  scanned 80,000 rows (2,118/s) | kept 11,693
  scanned 100,000 rows (2,209/s) | kept 14,755
  scanned 120,000 rows (2,277/s) | kept 17,782
  scanned 140,000 rows (2,326/s) | kept 20,526
  scanned 160,000 rows (2,364/s) | kept 23,806
  scanned 180,000 rows (2,393/s) | kept 25,380
  scanned 200,000 rows (2,420/s) | kept 28,166
  scanned 220,000 rows (2,443/s) | kept 31,248
  scanned 240,000 rows (2,462/s) | kept 34,235
  scanned 260,000 rows (2,479/s) | kept 37,589
  scanned 280,000 rows (2,493/s) | kept 40,044
  scanned 300,000 rows (2,504/s) | kept 42,816
  scanned 320,000 rows (2,515/s) | kept 45,730
  scanned 340,000 rows (2,525/s) | kept 48,073
  scan

In [12]:
# Checkpoint to a PRIVATE HF repo (Kaggle's disk disappears when the session ends)
!python -m rabbithole upload --file /kaggle/working/data/rabbithole.db --in-repo rabbithole.db --repo {CKPT_REPO} --private

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...orking/data/rabbithole.db:   0%|              |  824kB /  280MB            

Processing Files (0 / 1)      :   0%|              |  824kB /  280MB, 80.0kB/s  
New Data Upload               :   1%|▏             |  824kB / 67.1MB, 80.0kB/s  

  ...orking/data/rabbithole.db:   0%|              |  824kB /  280MB            

Processing Files (0 / 1)      :   1%|              | 1.65MB /  280MB,  158kB/s  
New Data Upload               :   2%|▎             | 1.65MB / 67.1MB,  158kB/s  

Processing Files (0 / 1)      :   1%|▏             | 4.12MB /  280MB,  394kB/s  
New Data Upload               :   6%|▊             | 4.12MB / 67.1MB,  394kB/s  

Processing Files (0 / 1)      :   2%|▎             | 5.77MB /  280MB,  549kB/s  
New Data Upload               :   4%|▌             | 5.77MB /  134MB,  549kB/s  

Processing Files (0 /

### B2. Thread metadata (real OP + title) → Q&A extraction

In [13]:
!python -m rabbithole meta

9,710 threads (>= 10 comments) need metadata
  50/9,710 done (found 50, missing 0, failed 0)
  1,050/9,710 done (found 1,050, missing 0, failed 0)
  2,050/9,710 done (found 2,050, missing 0, failed 0)
  3,050/9,710 done (found 3,050, missing 0, failed 0)
  4,050/9,710 done (found 4,050, missing 0, failed 0)
  5,050/9,710 done (found 5,045, missing 5, failed 0)
  6,050/9,710 done (found 5,995, missing 5, failed 50)
  7,050/9,710 done (found 6,995, missing 5, failed 50)
  8,050/9,710 done (found 7,995, missing 5, failed 50)
  9,050/9,710 done (found 8,995, missing 5, failed 50)
Finished. found=9,655 missing=5 failed(will retry on re-run)=50


In [14]:
!python -m rabbithole extract
!python -m rabbithole stats

9,655 threads have submission metadata
  500 threads -> pairs so far
  1,000 threads -> pairs so far
  1,500 threads -> pairs so far
  2,000 threads -> pairs so far
  2,500 threads -> pairs so far
  3,000 threads -> pairs so far
  3,500 threads -> pairs so far
  4,000 threads -> pairs so far
  4,500 threads -> pairs so far
  5,000 threads -> pairs so far
  5,500 threads -> pairs so far
{
  "threads_seen": 16915,
  "threads_no_pairs": 3798,
  "threads_with_pairs": 5912,
  "threads_too_small": 7205
}
Extracted 64,600 unique Q&A pairs
pairs: 64,600
threads: 5912
op_source: {'submission': 64600}
categories:
  other                    18,167  (28.1%)
  health_medicine           7,603  (11.8%)
  relationships_family      4,958  (7.7%)
  education                 4,868  (7.5%)
  entertainment_media       4,784  (7.4%)
  identity_lifestyle        4,090  (6.3%)
  crime_law                 3,337  (5.2%)
  tech                      2,952  (4.6%)
  jobs_trades               2,580  (4.0%)
  busines

**Look at the stats before indexing.** Total pairs, number of threads, `op_source`, category spread, biggest threads.
Write the pair/thread counts down — they're your new headline numbers for the README.

### B3. Build the search index (needs GPU for speed)

In [15]:
!python -m rabbithole index

config_sentence_transformers.json: 100%|████████| 124/124 [00:00<00:00, 590kB/s]
README.md: 100%|███████████████████████████| 94.8k/94.8k [00:00<00:00, 91.1MB/s]
sentence_bert_config.json: 100%|██████████████| 52.0/52.0 [00:00<00:00, 152kB/s]
config.json: 100%|█████████████████████████████| 743/743 [00:00<00:00, 2.91MB/s]

model.safetensors: downloading bytes: ██████████▌           | 63.9MB, 4.33MB/s  
model.safetensors: reconstructing file:  80%|████▊ |  106MB /  133MB            
model.safetensors: downloading bytes: ██████████████████████| 86.7MB, 8.20MB/s  
model.safetensors: reconstructing file: 100%|██████|  133MB /  133MB, 12.8MB/s  
tokenizer_config.json: 100%|███████████████████| 366/366 [00:00<00:00, 1.01MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 22.4MB/s]
tokenizer.json: 100%|████████████████████████| 711k/711k [00:00<00:00, 76.8MB/s]
config.json: 100%|██████████████████████████████| 190/190 [00:00<00:00, 806kB/s]
/kaggle/working/reddit-rabb

In [16]:
# Publish artifacts so the HF Space (and any other machine) can download them
!python -m rabbithole upload --repo {ARTIFACTS_REPO}

Found 11 files to upload
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files  524kB · 1.05MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files  24.4MB · 15.0MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files  105MB · 59.1MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 8 files  133MB · 58.1MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 11
  Preparing   ████████████████████  11 / 11 ✓
  Uploading   ██░░░░░░░░░░░░░░░░░░

## C. Try it
Real embeddings + cross-encoder reranker. The first call downloads the models.

In [17]:
!python -m rabbithole search "What do people in prison actually do all day?" -k 4

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2709.06it/s]
/kaggle/working/reddit-rabbithole-rag/rabbithole/embed.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())
config.json: 100%|█████████████████████████████| 794/794 [00:00<00:00, 3.50MB/s]

model.safetensors: downloading bytes: █████████████████▉    | 73.9MB, 6.05MB/s  
model.safetensors: downloading bytes: ██████████████████████| 86.0MB, 8.23MB/s  
model.safetensors: reconstructing file: 100%|██████| 90.9MB / 90.9MB, 8.84MB/s  
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2519.00it/s]
tokenizer_config.json: 100%|███████████████| 1.33k/1.33k [00:00<00:00, 3.00MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 16.8MB/s]
tokenizer.json: 100%|████████████████████████| 711k/711k [00:00<00:00, 14.7MB/s]
special_tokens_map.json: 100%|██████

In [18]:
# Needs GEMINI_API_KEY. If you get a 404, run the next cell and set GEMINI_MODEL.
!python -m rabbithole ask "What is the hardest part of being a nurse?"

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2734.82it/s]
/kaggle/working/reddit-rabbithole-rag/rabbithole/embed.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2775.34it/s]
The available AMAs don't cover this directly. 

citations: CitationReport(cited=[], invalid=[], sentences=1, uncited_sentences=0, abstained=True)
[1] I am 20 and work at a nursing home for alzheimers patients who are competely non  https://www.reddit.com/comments/aora1/_/c0inpyg/
[2] IAmA: Multiple Personality  https://www.reddit.com/comments/goujz/_/c1p5yop/
[3] IAmA NYC nurse in a major trauma center in NYC. AMA  https://www.reddit.com/comments/lxizp/_/c2webaz/
[4] IAmA Psychiatric Nurse who works in a mental institute. AMA  https://www.reddit.com/comments/kyu5x/_/c2oduef/


In [19]:
!python -m rabbithole gemini-models

gemini-2.5-flash
gemini-2.5-flash-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts


## D. Evaluate (this produces the numbers for your README)

In [20]:
!python -m rabbithole eval-make -n 200

  25/200 queries
  skip: Gemini request failed (gemini-flash-latest, gemini-2.5-flash): 429: {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota ex
  skip: Gemini request failed (gemini-flash-latest, gemini-2.5-flash): 429: {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota ex
  skip: Gemini request failed (gemini-flash-latest, gemini-2.5-flash): 429: {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this

In [21]:
!python -m rabbithole eval-retrieval

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2793.06it/s]
/kaggle/working/reddit-rabbithole-rag/rabbithole/embed.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2650.98it/s]
| Mode | R@1 | R@5 | R@10 | R@20 | MRR@20 | Thread R@5 | Latency ms (mean / p95) |
|---|---|---|---|---|---|---|---|
| bm25 | 0.308 | 0.615 | 0.641 | 0.667 | 0.436 | 0.667 | 4 / 3 |
| dense | 0.462 | 0.641 | 0.744 | 0.821 | 0.555 | 0.769 | 28 / 22 |
| hybrid | 0.513 | 0.667 | 0.692 | 0.744 | 0.573 | 0.718 | 23 / 26 |
| hybrid_rerank | 0.615 | 0.692 | 0.769 | 0.769 | 0.653 | 0.795 | 100 / 119 |

Saved to eval/results/retrieval.md


In [22]:
!python -m rabbithole calibrate

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2842.27it/s]
/kaggle/working/reddit-rabbithole-rag/rabbithole/embed.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2757.68it/s]
{
  "in_domain_top_score": {
    "p10": -0.9027090072631836,
    "median": 2.4619932174682617,
    "p90": 5.618965148925781
  },
  "ood_top_score": {
    "p10": -6.969519138336182,
    "median": -4.767239093780518,
    "p90": -2.0680251121520996
  },
  "suggested_threshold": -0.9027090072631836,
  "balanced_accuracy": 0.9666666666666667,
  "n_in": 30,
  "n_ood": 12
}

Set RR_ABSTAIN_THRESHOLD=-0.9027090072631836 (small seed set: sanity-check before trusting).


`calibrate` prints a `suggested_threshold`. Sanity-check it (the seed set is small), then set it as the Space variable `RR_ABSTAIN_THRESHOLD`.

In [23]:
!python -m rabbithole eval-generation --judge

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2750.46it/s]
/kaggle/working/reddit-rabbithole-rag/rabbithole/embed.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2793.26it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/reddit-rabbithole-rag/rabbithole/__main__.py", line 4, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/reddit-rabbithole-rag/rabbithole/cli.py", line 158, in main
    out = eval_generation(build_pipeline(s), seed["in_domain"], seed["out_of_domain"], judge)
  File "/kaggle/working/reddit-rabbithole-rag/rabbithole/eval.py", line 172, in eval_generation
    a, b = run(in_domain), run(ood)
           ~~~^^^^^^^^^^^
  File "/kaggle

In [24]:
# Show the retrieval table to paste into the README
print(open("eval/results/retrieval.md").read())

| Mode | R@1 | R@5 | R@10 | R@20 | MRR@20 | Thread R@5 | Latency ms (mean / p95) |
|---|---|---|---|---|---|---|---|
| bm25 | 0.308 | 0.615 | 0.641 | 0.667 | 0.436 | 0.667 | 4 / 3 |
| dense | 0.462 | 0.641 | 0.744 | 0.821 | 0.555 | 0.769 | 28 / 22 |
| hybrid | 0.513 | 0.667 | 0.692 | 0.744 | 0.573 | 0.718 | 23 / 26 |
| hybrid_rerank | 0.615 | 0.692 | 0.769 | 0.769 | 0.653 | 0.795 | 100 / 119 |

_n = 39 synthetic queries_



## E. Wrap up
- Artifacts are on `ARTIFACTS_REPO` → deploy the Space with `deploy/push_space.sh` and set secrets `GEMINI_API_KEY`, `ARTIFACTS_REPO` (+ `RR_ABSTAIN_THRESHOLD`).
- Download `eval/results/*` from the Output tab and paste the table into the README.
- Rotate any key that was ever pasted into a notebook.